import json
from pathlib import Path
import sqlite3
import math

import pandas as pd
import shutil

BASE_DIR = Path('C:/Users/Aarush/Documents/FRC/2026Scripts')
MATCH_JSON = BASE_DIR / 'betterSB' / 'Caismatchmath.json'
PICKLIST_CSV = BASE_DIR / 'caisTeam.csv'
MATCH_DB = BASE_DIR / 'match.db'

CLONE_DB = MATCH_DB.with_name('2' + MATCH_DB.name)
shutil.copy2(MATCH_DB, CLONE_DB)
print('Cloned database to', CLONE_DB)

print('JSON path:', MATCH_JSON)
print('Picklist:', PICKLIST_CSV)
print('Database:', MATCH_DB)


def safe_number(value):
    if value is None:
        return None
    if isinstance(value, (int, float)):
        if isinstance(value, float) and math.isnan(value):
            return None
        return float(value)
    try:
        num = float(value)
    except (TypeError, ValueError):
        return None
    if math.isnan(num):
        return None
    return num


def compute_time(points, bps, accuracy):
    points = safe_number(points)
    bps = safe_number(bps)
    accuracy = safe_number(accuracy)
    if points is None or bps is None or accuracy is None:
        return None
    denom = bps * accuracy
    if denom == 0 or math.isnan(denom):
        return None
    return points / denom


PHASE_SCORING_KEYS = {
    'auto': 'auto_points',
    'teleop': 'tele_points',
}


def get_selected_points(team_record, phase):
    selected_key = f'selected_{phase}'
    selected_value = team_record.get(selected_key)
    if selected_value is not None:
        return selected_value
    scouting_points = team_record.get('points', {}).get('scouting', {})
    fallback_key = PHASE_SCORING_KEYS.get(phase)
    if fallback_key:
        return scouting_points.get(fallback_key)
    return None


In [12]:
import json
from pathlib import Path
import sqlite3
import math

import pandas as pd
import shutil

BASE_DIR = Path('C:/Users/Aarush/Documents/FRC/2026Scripts')
MATCH_JSON = BASE_DIR / 'betterSB' / 'Caismatchmath.json'
PICKLIST_CSV = BASE_DIR / 'caisTeam.csv'
MATCH_DB = BASE_DIR / 'match.db'

CLONE_DB = MATCH_DB.with_name('2' + MATCH_DB.name)
shutil.copy2(MATCH_DB, CLONE_DB)
print('Cloned database to', CLONE_DB)

print('JSON path:', MATCH_JSON)
print('Picklist:', PICKLIST_CSV)
print('Database:', MATCH_DB)


def safe_number(value):
    if value is None:
        return None
    if isinstance(value, (int, float)):
        if isinstance(value, float) and math.isnan(value):
            return None
        return float(value)
    try:
        num = float(value)
    except (TypeError, ValueError):
        return None
    if math.isnan(num):
        return None
    return num


def compute_time(points, bps, accuracy):
    points = safe_number(points)
    bps = safe_number(bps)
    accuracy = safe_number(accuracy)
    if points is None or bps is None or accuracy is None:
        return None
    denom = bps * accuracy
    if denom == 0 or math.isnan(denom):
        return None
    return points / denom


PHASE_SCORING_KEYS = {
    'auto': 'auto_points',
    'teleop': 'tele_points',
}


def get_selected_points(team_record, phase):
    selected_key = f'selected_{phase}'
    selected_value = team_record.get(selected_key)
    if selected_value is not None:
        return selected_value
    scouting_points = team_record.get('points', {}).get('scouting', {})
    fallback_key = PHASE_SCORING_KEYS.get(phase)
    if fallback_key:
        return scouting_points.get(fallback_key)
    return None


with MATCH_JSON.open() as match_file:
    match_data = json.load(match_file)
matches = match_data.get('matches') or []
print(f'Loaded {len(matches)} matches from {MATCH_JSON.name}')

picklist = {}
picklist_df = pd.read_csv(PICKLIST_CSV)
for record in picklist_df.to_dict('records'):
    team_key = record.get('team_key') or record.get('key')
    if not isinstance(team_key, str):
        continue
    team_key = team_key.strip().lower()
    if not team_key:
        continue
    metrics = {
        'auto_bps': safe_number(record.get('bps_1')),
        'tele_bps': None,
        'auto_acc': safe_number(record.get('accuracy_1')),
        'tele_acc': None,
    }
    if metrics['auto_bps'] is None and metrics['auto_acc'] is None:
        continue
    picklist[team_key] = metrics

print(
    f'Loaded scoring-rate metrics for {len(picklist)} teams',
    f'from {PICKLIST_CSV.name}'
)



Cloned database to C:\Users\Aarush\Documents\FRC\2026Scripts\2match.db
JSON path: C:\Users\Aarush\Documents\FRC\2026Scripts\betterSB\Caismatchmath.json
Picklist: C:\Users\Aarush\Documents\FRC\2026Scripts\caisTeam.csv
Database: C:\Users\Aarush\Documents\FRC\2026Scripts\match.db
Loaded 52 matches from Caismatchmath.json
Loaded scoring-rate metrics for 35 teams from caisTeam.csv


In [13]:
updates = []
missing_picklist = set()

for match_record in matches:
    match_key = match_record['match_key']
    for team_record in match_record['teams']:
        team_key = team_record['team_key']
        info = picklist.get(team_key)
        if info is None:
            missing_picklist.add(team_key)
            continue
        auto_points = get_selected_points(team_record, 'auto')
        tele_points = get_selected_points(team_record, 'teleop')
        auto_time = compute_time(auto_points, info['auto_bps'], info['auto_acc'])
        tele_time = compute_time(tele_points, info['tele_bps'], info['tele_acc'])
        if auto_time is None and tele_time is None:
            continue
        updates.append((auto_time, tele_time, match_key, team_key))

print(
    f'Prepared {len(updates)} scoring-time updates.',
    f'Missing picklist info for {len(missing_picklist)} teams.'
)
if missing_picklist:
    print('Examples of missing teams:', ', '.join(sorted(missing_picklist))[:200])

sql = '''
UPDATE matches
SET
    a_scoringTime = CASE WHEN ? IS NOT NULL THEN ? ELSE a_scoringTime END,
    t_scoringTime = CASE WHEN ? IS NOT NULL THEN ? ELSE t_scoringTime END
WHERE key = ? AND team_key = ?
'''

conn = sqlite3.connect(MATCH_DB)
cursor = conn.cursor()
rows_affected = 0
for auto_time, tele_time, match_key, team_key in updates:
    cursor.execute(sql, (auto_time, auto_time, tele_time, tele_time, match_key, team_key))
    rows_affected += cursor.rowcount
conn.commit()
conn.close()
print('Rows affected:', rows_affected)


Prepared 312 scoring-time updates. Missing picklist info for 0 teams.
Rows affected: 312


In [14]:
with sqlite3.connect(MATCH_DB) as conn:
    cursor = conn.cursor()
    samples = []
    for match_record in matches[:1]:
        match_key = match_record['match_key']
        for team_record in match_record['teams'][:6]:
            team_key = team_record['team_key']
            info = picklist.get(team_key)
            if info is None:
                continue
            auto_points = get_selected_points(team_record, 'auto')
            tele_points = get_selected_points(team_record, 'teleop')
            auto_time = compute_time(auto_points, info['auto_bps'], info['auto_acc'])
            tele_time = compute_time(tele_points, info['tele_bps'], info['tele_acc'])
            cursor.execute(
                'SELECT a_scoringTime, t_scoringTime FROM matches WHERE key = ? AND team_key = ?',
                (match_key, team_key),
            )
            row = cursor.fetchone()
            samples.append({
                'match_key': match_key,
                'team_key': team_key,
                'selected_auto': team_record.get('selected_auto'),
                'selected_teleop': team_record.get('selected_teleop'),
                'auto_points': auto_points,
                'tele_points': tele_points,
                'auto_bps': info['auto_bps'],
                'auto_acc': info['auto_acc'],
                'tele_bps': info['tele_bps'],
                'tele_acc': info['tele_acc'],
                'computed_auto_time': auto_time,
                'computed_tele_time': tele_time,
                'db_auto_time': row[0],
                'db_tele_time': row[1],
            })

sample_df = pd.DataFrame(samples)
print('Sample of updated scoring times (first match, six teams):')
print(sample_df)


Sample of updated scoring times (first match, six teams):
       match_key  team_key  selected_auto  selected_teleop  auto_points  \
0  2026orore_qm1   frc2635         9.5800         38.73856       9.5800   
1  2026orore_qm1   frc2557        28.4475         14.35000      28.4475   
2  2026orore_qm1   frc3669        10.6700         26.13000      10.6700   
3  2026orore_qm1   frc2898         2.6400         12.13641       2.6400   
4  2026orore_qm1  frc10991         7.4900          0.00000       7.4900   
5  2026orore_qm1   frc8032        20.1664        141.47300      20.1664   

   tele_points  auto_bps  auto_acc tele_bps tele_acc  computed_auto_time  \
0     38.73856     2.240      1.00     None     None            4.276786   
1     14.35000    10.000      0.75     None     None            3.793000   
2     26.13000     1.000      1.00     None     None           10.670000   
3     12.13641     5.060      0.78     None     None            0.668896   
4      0.00000     3.235      1.00  